# 02_generative_ui_and_tools

Moving beyond basic text streaming, enterprise GenAI applications require bridging the gap between raw tool outputs and visual, interactive user interfaces. This is the domain of Generative UI (GenUI) and tool-driven state rendering.

Hiring managers look for engineers who understand how to transform asynchronous backend tool calls into dynamic front-end components safely, avoiding security vulnerabilities like UI injection and XSS.

## 1. Core Concepts & Architectural Patterns
The Tool-to-UI Lifecycle Contract:Instead of hardcoding every possible layout state, modern GenAI apps use a structured contract where the LLM does not write code—it selects a pre-vetted component from a Catalog and passes structured arguments (JSON) to configure it.

The Three GenUI Approaches:

 1. Static / Controlled GenUI: The model chooses from a strict library of hand-built React components (guarantees brand safety and strict styling).  

 2. Declarative GenUI: The model returns a typed schema or protocol data payload that maps to modular UI blocks.  

 3. Open-Ended GenUI (High Risk): The model emits raw markup or code evaluated dynamically on the client. Production architectures strictly avoid this due to severe XSS risks.  

Trust Boundaries & Security:If a tool execution fetches untrusted external data (e.g., scraped data containing indirect prompt injections), rendering that raw data directly as component code can hijack the UI or trigger cross-site scripting. Production systems enforce strict schema validation via Zod / Pydantic contracts between the tool output and the UI props.  

## 2. Production Implementation Pattern (TypeScript / React Component Mapping)
Here is a clean frontend architectural pattern showing how a client handles tool execution states and maps tool results directly to secure, structured generative widgets.

In [ ]:
import React from 'react';

// 1. Define strict type contracts for component props
interface WeatherCardProps {
  location: string;
  temperature: string;
  unit: string;
}

function WeatherWidget({ location, temperature, unit }: WeatherCardProps) {
  return (
    <div className="p-4 my-2 bg-gradient-to-r from-blue-50 to-indigo-50 border border-blue-200 rounded-xl shadow-sm max-w-sm">
      <div className="flex justify-between items-center">
        <span className="text-xs font-bold tracking-wider text-blue-600 uppercase">Live Weather Tool</span>
        <span className="text-2xl">⛅</span>
      </div>
      <h3 className="text-lg font-semibold text-slate-900 mt-1">{location}</h3>
      <p className="text-3xl font-extrabold text-blue-900 mt-2">{temperature}° {unit}</p>
    </div>
  );
}

// 2. Main GenUI Surface Dispatcher
export function GenerativeSurface({ toolInvocation }: { toolInvocation: any }) {
  const { toolName, state, args, result } = toolInvocation;

  // Lifecycle State 1: Tool is actively executing on the backend
  if (state === 'call') {
    return (
      <div className="flex items-center space-x-3 p-3 my-2 bg-slate-50 border border-slate-200 rounded-lg text-xs text-slate-600 animate-pulse">
        <div className="w-3 h-3 bg-blue-500 rounded-full animate-ping" />
        <span>Executing tool: <strong className="font-mono">{toolName}</strong>...</span>
      </div>
    );
  }

  // Lifecycle State 2: Tool execution complete, render secure Generative UI component
  if (state === 'result') {
    switch (toolName) {
      case 'get_current_weather':
        // Safely pass parsed and validated arguments/results into typed components
        return (
          <WeatherWidget 
            location={args.location} 
            temperature={result.temperature} 
            unit={result.unit} 
          />
        );
        
      default:
        return (
          <div className="p-3 my-2 bg-red-50 border border-red-200 rounded-lg text-xs text-red-700">
            Unknown tool output mapping for: {toolName}
          </div>
        );
    }
  }

  return null;
}

## 3. Deep-Dive: Architecture & Trade-offs
Handling Non-Deterministic UI Failures: Traditional frontend components break if expected JSON properties are missing. Because LLMs can occasionally return malformed or partial tool arguments, your UI mapper must implement defensive fallbacks (e.g., Zod safe parsing or default fallback states) so that a bad tool payload degrades gracefully into a text fallback instead of crashing the entire React error boundary.

Accessibility (A11y) in GenUI: People at accessibility-conscious enterprises will ask how dynamic components maintain WCAG compliance. Emphasize that because components are pre-compiled and design-system managed (rather than built out of thin air by the LLM), accessibility roles, ARIA labels, and keyboard navigability are baked directly into the base component wrappers.